# Step 4: Feature Extraction
**INT 396 — Unsupervised Learning | Social Media Topic Discovery for Disaster Response**

Generate two representations for each tweet:
- **Approach A:** SBERT embeddings (all-MiniLM-L6-v2, 384-dim dense vectors)
- **Approach B:** TF-IDF vectors (max 10,000 features, bigrams included)

Both applied to HumAID (full) and CrisisBench validation event subsets.

In [1]:
import os
from pathlib import Path

import numpy as np
import pandas as pd
import pickle
from sklearn.feature_extraction.text import TfidfVectorizer
from sentence_transformers import SentenceTransformer
import scipy.sparse
import time

PROJECT_ROOT = Path.cwd().resolve()
if not (PROJECT_ROOT / "requirements.txt").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
PROJECT_ROOT = str(PROJECT_ROOT)
os.makedirs(f"{PROJECT_ROOT}/data/embeddings", exist_ok=True)

# Load cleaned datasets
humaid = pd.read_parquet(f"{PROJECT_ROOT}/data/processed/humaid_clean.parquet")
print(f"HumAID: {len(humaid):,} rows")

# Load CrisisBench validation subsets
cb_nepal = pd.read_parquet(f"{PROJECT_ROOT}/data/processed/cb_2015_nepal_earthquake.parquet")
cb_harvey = pd.read_parquet(f"{PROJECT_ROOT}/data/processed/cb_hurricane_harvey.parquet")
cb_alberta = pd.read_parquet(f"{PROJECT_ROOT}/data/processed/cb_2013_alberta_floods.parquet")
print(f"CrisisBench subsets: Nepal={len(cb_nepal):,}, Harvey={len(cb_harvey):,}, Alberta={len(cb_alberta):,}")

/Users/cyril/Desktop/Un-supervised/.venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


HumAID: 76,477 rows
CrisisBench subsets: Nepal=5,236, Harvey=3,022, Alberta=4,159


## 4.1 Approach A — SBERT Embeddings (all-MiniLM-L6-v2, 384-dim)

In [2]:
# Load SBERT model
model = SentenceTransformer("all-MiniLM-L6-v2")
print(f"Model: all-MiniLM-L6-v2, output dim: {model.get_sentence_embedding_dimension()}")

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 8607.04it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Model: all-MiniLM-L6-v2, output dim: 384


In [3]:
# Encode HumAID
print("Encoding HumAID...")
start = time.time()
humaid_sbert = model.encode(humaid["text_clean"].tolist(), show_progress_bar=True, batch_size=256)
elapsed = time.time() - start
print(f"Done in {elapsed:.1f}s — shape: {humaid_sbert.shape}")

np.save(f"{PROJECT_ROOT}/data/embeddings/humaid_sbert.npy", humaid_sbert)
print("Saved: humaid_sbert.npy")

Encoding HumAID...


Batches: 100%|██████████| 299/299 [02:07<00:00,  2.34it/s]


Done in 128.6s — shape: (76477, 384)
Saved: humaid_sbert.npy


In [4]:
# Encode CrisisBench validation subsets
for name, df in [("cb_nepal", cb_nepal), ("cb_harvey", cb_harvey), ("cb_alberta", cb_alberta)]:
    print(f"Encoding {name}...")
    start = time.time()
    emb = model.encode(df["text_clean"].tolist(), show_progress_bar=True, batch_size=256)
    elapsed = time.time() - start
    print(f"  {emb.shape} in {elapsed:.1f}s")
    np.save(f"{PROJECT_ROOT}/data/embeddings/{name}_sbert.npy", emb)

print("\nAll SBERT embeddings saved.")

Encoding cb_nepal...


Batches: 100%|██████████| 21/21 [00:12<00:00,  1.75it/s]


  (5236, 384) in 12.2s
Encoding cb_harvey...


Batches: 100%|██████████| 12/12 [00:04<00:00,  2.73it/s]


  (3022, 384) in 4.4s
Encoding cb_alberta...


Batches: 100%|██████████| 17/17 [00:08<00:00,  1.99it/s]

  (4159, 384) in 8.6s

All SBERT embeddings saved.


## 4.2 Approach B — TF-IDF Vectors (max 10,000 features, bigrams)

In [5]:
# Fit TF-IDF on HumAID (primary dataset)
tfidf = TfidfVectorizer(
    max_features=10000,
    ngram_range=(1, 2),
    min_df=3,
    max_df=0.95,
    sublinear_tf=True
)

print("Fitting TF-IDF on HumAID...")
start = time.time()
humaid_tfidf = tfidf.fit_transform(humaid["text_clean"])
elapsed = time.time() - start
print(f"Done in {elapsed:.1f}s — shape: {humaid_tfidf.shape}")
print(f"Vocabulary size: {len(tfidf.vocabulary_):,}")

# Save TF-IDF matrix and fitted vectorizer
scipy.sparse.save_npz(f"{PROJECT_ROOT}/data/embeddings/humaid_tfidf.npz", humaid_tfidf)
with open(f"{PROJECT_ROOT}/data/embeddings/tfidf_vectorizer.pkl", "wb") as f:
    pickle.dump(tfidf, f)
print("Saved: humaid_tfidf.npz + tfidf_vectorizer.pkl")

Fitting TF-IDF on HumAID...
Done in 6.1s — shape: (76477, 10000)
Vocabulary size: 10,000
Saved: humaid_tfidf.npz + tfidf_vectorizer.pkl


In [6]:
# Transform CrisisBench validation subsets using the HumAID-fitted vectorizer
for name, df in [("cb_nepal", cb_nepal), ("cb_harvey", cb_harvey), ("cb_alberta", cb_alberta)]:
    print(f"Transforming {name}...")
    X = tfidf.transform(df["text_clean"])
    print(f"  {X.shape}")
    scipy.sparse.save_npz(f"{PROJECT_ROOT}/data/embeddings/{name}_tfidf.npz", X)

print("\nAll TF-IDF vectors saved.")

Transforming cb_nepal...
  (5236, 10000)
Transforming cb_harvey...
  (3022, 10000)
Transforming cb_alberta...
  (4159, 10000)

All TF-IDF vectors saved.


## 4.3 Quick Sanity Check

In [7]:
# Verify all saved files
print("Saved embeddings:")
for f in sorted(os.listdir(f"{PROJECT_ROOT}/data/embeddings")):
    path = os.path.join(f"{PROJECT_ROOT}/data/embeddings", f)
    size_mb = os.path.getsize(path) / (1024 * 1024)
    print(f"  {f}: {size_mb:.1f} MB")

# Quick similarity check on SBERT
from sklearn.metrics.pairwise import cosine_similarity
sample_texts = [
    "earthquake destroyed many buildings",
    "buildings collapsed in the earthquake",
    "donations needed for flood victims",
    "please donate to help flood relief"
]
sample_emb = model.encode(sample_texts)
sim_matrix = cosine_similarity(sample_emb)

print("\nSBERT cosine similarity (sanity check):")
for i, t1 in enumerate(sample_texts):
    for j, t2 in enumerate(sample_texts):
        if j > i:
            print(f"  '{t1[:40]}' vs '{t2[:40]}': {sim_matrix[i][j]:.3f}")

Saved embeddings:
  cb_alberta_sbert.npy: 6.1 MB
  cb_alberta_tfidf.npz: 0.6 MB
  cb_harvey_sbert.npy: 4.4 MB
  cb_harvey_tfidf.npz: 0.3 MB
  cb_nepal_sbert.npy: 7.7 MB
  cb_nepal_tfidf.npz: 0.6 MB
  humaid_sbert.npy: 112.0 MB
  humaid_tfidf.npz: 16.6 MB
  tfidf_vectorizer.pkl: 0.4 MB

SBERT cosine similarity (sanity check):
  'earthquake destroyed many buildings' vs 'buildings collapsed in the earthquake': 0.832
  'earthquake destroyed many buildings' vs 'donations needed for flood victims': 0.218
  'earthquake destroyed many buildings' vs 'please donate to help flood relief': 0.194
  'buildings collapsed in the earthquake' vs 'donations needed for flood victims': 0.204
  'buildings collapsed in the earthquake' vs 'please donate to help flood relief': 0.181
  'donations needed for flood victims' vs 'please donate to help flood relief': 0.834


## 4.4 Summary

Feature extraction complete:
- **SBERT** (384-dim dense): HumAID + 3 CrisisBench events → `.npy` files
- **TF-IDF** (10K sparse): HumAID + 3 CrisisBench events → `.npz` files
- Vectorizer saved as `.pkl` for reproducibility

**Next step:** Run `05_dimensionality_reduction.ipynb` for UMAP (with grid search), PCA, TruncatedSVD, and t-SNE visualization.